In [ ]:
# ============================================================
# 03_model_training_evaluation.ipynb
# Entraînement Prophet + évaluation des performances
# ============================================================

# ── Cellule 1 : Configuration ────────────────────────────────
BASE_NAME = 'BIJOU'   # ← changer ici

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from prophet import Prophet
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

HORIZON_PREVISION = 30
TEST_SPLIT_DAYS   = 30

os.makedirs(f'../models/{BASE_NAME}', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

print(f"✅ Config OK — {BASE_NAME}")
print(f"   Horizon prévision : {HORIZON_PREVISION} jours")

In [ ]:
# ── Cellule 2 : Chargement dataset préparé ───────────────────
df = pd.read_parquet(f'../outputs/{BASE_NAME}_dataset_features.parquet')
df['DateJour'] = pd.to_datetime(df['DateJour'])

date_max   = df['DateJour'].max()
date_split = date_max - pd.Timedelta(days=TEST_SPLIT_DAYS)

print(f"date_max   : {date_max.date()}")
print(f"date_split : {date_split.date()}")
print(f"Période test : {TEST_SPLIT_DAYS} derniers jours")

# Liste des combinaisons à entraîner
combos = df.groupby(['AR_Ref', 'DE_No']).size().reset_index(name='n')
print(f"   Combinaisons à entraîner : {len(combos)}")
combos.head()

In [ ]:
# ── Cellule 2b (à ajouter après le chargement) ────────────────
# Compter les jours AVEC mouvement réel par combo
# (pas les jours reportés où TotalEntree=0 et TotalSortie=0)

jours_avec_mvt = (
    df[( df['TotalEntree'] > 0) | (df['TotalSortie'] > 0)]
    .groupby(['AR_Ref', 'DE_No'])
    .size()
    .reset_index(name='nb_jours_mvt_reel')
)

combos = df.groupby(['AR_Ref', 'DE_No']).size().reset_index(name='n_total')
combos = combos.merge(jours_avec_mvt, on=['AR_Ref', 'DE_No'], how='left').fillna(0)

# Filtrer : au moins 60 jours avec mouvement réel
combos_valides = combos[combos['nb_jours_mvt_reel'] >= 60]

print(f"Combinaisons total     : {len(combos)}")
print(f"Combinaisons valides   : {len(combos_valides)}")

# Filtrer le dataset
df = df[df.set_index(['AR_Ref','DE_No']).index.isin(
    combos_valides.set_index(['AR_Ref','DE_No']).index
)].reset_index(drop=True)

In [ ]:
# ── Cellule 3 : Fonction d'entraînement Prophet ──────────────
def entrainer_prophet(df_combo, ar_ref, de_no):
    """
    Entraîne un modèle Prophet sur les sorties d'une combinaison.
    Retourne (model, metrics_dict) ou (None, None) si données insuffisantes.
    """
    # Format Prophet : colonnes ds (date) et y (valeur cible)
    df_prophet = df_combo[['DateJour', 'TotalSortie']].copy()
    df_prophet.columns = ['ds', 'y']
    df_prophet = df_prophet.sort_values('ds').reset_index(drop=True)
    
    # Vérification : minimum 60 jours
    if len(df_prophet) < 60:
        return None, None
    
    # Split train/test
    train = df_prophet[df_prophet['ds'] <= date_split]
    test  = df_prophet[df_prophet['ds'] >  date_split]
    
    if len(train) < 30 or len(test) == 0:
        return None, None
    
    # Modèle Prophet
    # daily_seasonality=False car déjà capturé par weekly
    # changepoint_prior_scale : contrôle la flexibilité de la tendance
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
        interval_width=0.80
    )
    
    # Ajouter les jours fériés marocains si pertinent
    # model.add_country_holidays(country_name='MA')
    
    model.fit(train)
    
    # Prédiction sur la période test
    future_test = model.make_future_dataframe(periods=len(test), freq='D')
    forecast_test = model.predict(future_test)
    
    # Métriques sur la période test uniquement
    pred_test = forecast_test[forecast_test['ds'].isin(test['ds'])][['ds', 'yhat']].copy()
    merged    = test.merge(pred_test, on='ds', how='inner')
    
    merged['yhat_clipped'] = merged['yhat'].clip(lower=0)  # sorties >= 0
    
    mae  = np.mean(np.abs(merged['y'] - merged['yhat_clipped']))
    rmse = np.sqrt(np.mean((merged['y'] - merged['yhat_clipped'])**2))
    
    y_mean = merged['y'].mean()
    mape = np.mean(np.abs((merged['y'] - merged['yhat_clipped']) / (y_mean + 1e-9))) * 100
    
    metrics = {
        'AR_Ref':  ar_ref,
        'DE_No':   de_no,
        'n_train': len(train),
        'n_test':  len(test),
        'MAE':     round(mae,  3),
        'RMSE':    round(rmse, 3),
        'MAPE':    round(mape, 2)
    }
    
    return model, metrics

In [ ]:
# ── Cellule 4 : Entraînement de tous les modèles ─────────────
all_metrics = []
modeles_ok  = 0
modeles_ko  = 0

print(f"⏳ Entraînement de {len(combos)} modèles...")
print("   (peut prendre plusieurs minutes selon le nombre d'articles)")

for idx, row in combos.iterrows():
    ar_ref = row['AR_Ref']
    de_no  = row['DE_No']
    
    df_combo = df[
        (df['AR_Ref'] == ar_ref) & (df['DE_No'] == de_no)
    ].sort_values('DateJour')
    
    model, metrics = entrainer_prophet(df_combo, ar_ref, de_no)
    
    if model is None:
        modeles_ko += 1
        continue
    
    # Sauvegarder le modèle
    safe_article = ar_ref.replace('/', '_').replace(' ', '_')
    model_path   = f'../models/{BASE_NAME}/{safe_article}_{de_no}.joblib'
    joblib.dump(model, model_path)
    
    all_metrics.append(metrics)
    modeles_ok += 1
    
    # Progression tous les 10 modèles
    if (modeles_ok + modeles_ko) % 10 == 0:
        print(f"   {modeles_ok + modeles_ko}/{len(combos)} — OK:{modeles_ok} KO:{modeles_ko}")

df_metrics = pd.DataFrame(all_metrics)

print(f"\n✅ Entraînement terminé")
print(f"   Modèles entraînés : {modeles_ok}")
print(f"   Modèles échoués   : {modeles_ko}")
print(f"   Modèles sauvegardés dans : ../models/{BASE_NAME}/")

In [ ]:
# ── Cellule 5 : Tableau des performances globales ────────────
print("=" * 55)
print(f"PERFORMANCES GLOBALES — {BASE_NAME}")
print("=" * 55)
print(f"\nMAE  moyenne  : {df_metrics['MAE'].mean():.3f}  (± {df_metrics['MAE'].std():.3f})")
print(f"RMSE moyenne  : {df_metrics['RMSE'].mean():.3f}  (± {df_metrics['RMSE'].std():.3f})")
print(f"MAPE moyenne  : {df_metrics['MAPE'].mean():.1f}%  (± {df_metrics['MAPE'].std():.1f}%)")

# Distribtution MAPE
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].hist(df_metrics['MAE'],  bins=20, color='#12a6e0', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution MAE', fontweight='bold')
axes[0].set_xlabel("MAE"); axes[0].set_ylabel("Nombre de modèles")

axes[1].hist(df_metrics['RMSE'], bins=20, color='#01a82e', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution RMSE', fontweight='bold')
axes[1].set_xlabel("RMSE")

axes[2].hist(df_metrics['MAPE'], bins=20, color='#e53935', edgecolor='white', alpha=0.85)
axes[2].set_title('Distribution MAPE (%)', fontweight='bold')
axes[2].set_xlabel("MAPE (%)")
axes[2].axvline(20, color='black', linestyle='--', linewidth=1.5, label='Seuil 20%')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'../outputs/{BASE_NAME}_09_performances_globales.png', dpi=150, bbox_inches='tight')
plt.show()

# Catégorisation des performances
df_metrics['qualite'] = pd.cut(
    df_metrics['MAPE'],
    bins=[-np.inf, 10, 20, 50, np.inf],
    labels=['Excellent (<10%)', 'Bon (10-20%)', 'Acceptable (20-50%)', 'Faible (>50%)']
)

print("\n📊 Qualité des prédictions :")
print(df_metrics['qualite'].value_counts().to_string())

In [ ]:
# ── Cellule 6 : Visualisation sur les 3 meilleurs articles ───
top3 = df_metrics.nsmallest(3, 'MAPE')

for _, perf in top3.iterrows():
    ar_ref = perf['AR_Ref']
    de_no  = perf['DE_No']
    
    safe_article = ar_ref.replace('/', '_').replace(' ', '_')
    model_path   = f'../models/{BASE_NAME}/{safe_article}_{de_no}.joblib'
    
    model  = joblib.load(model_path)
    
    df_combo = df[
        (df['AR_Ref'] == ar_ref) & (df['DE_No'] == de_no)
    ].sort_values('DateJour')
    
    df_prophet = df_combo[['DateJour', 'TotalSortie']].copy()
    df_prophet.columns = ['ds', 'y']
    
    # Prédiction sur toute la période + horizon futur
    future = model.make_future_dataframe(periods=HORIZON_PREVISION, freq='D')
    forecast = model.predict(future)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Données réelles
    ax.plot(df_prophet['ds'], df_prophet['y'],
            color='#333333', linewidth=1.2, label='Réel', zorder=3)
    
    # Zone train / test
    ax.axvspan(df_prophet['ds'].min(), date_split,
               alpha=0.05, color='#12a6e0', label='Période train')
    ax.axvspan(date_split, df_prophet['ds'].max(),
               alpha=0.08, color='#ff9800', label='Période test')
    
    # Prédiction
    pred_plot = forecast[forecast['ds'] >= date_split - pd.Timedelta(days=5)]
    ax.plot(pred_plot['ds'], pred_plot['yhat'].clip(lower=0),
            color='#e53935', linewidth=2, linestyle='--', label='Prédiction')
    
    ax.fill_between(
        pred_plot['ds'],
        pred_plot['yhat_lower'].clip(lower=0),
        pred_plot['yhat_upper'].clip(lower=0),
        alpha=0.15, color='#e53935', label='Intervalle confiance 80%'
    )
    
    design = df_combo['AR_Design'].iloc[0][:30] if 'AR_Design' in df_combo.columns else ''
    ax.set_title(
        f'Prédiction — Article {ar_ref} ({design}) | Dépôt {de_no}\n'
        f'MAE={perf["MAE"]:.2f}  RMSE={perf["RMSE"]:.2f}  MAPE={perf["MAPE"]:.1f}%',
        fontweight='bold', fontsize=11
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Sorties (unités)")
    ax.legend(loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(f'../outputs/{BASE_NAME}_10_pred_{safe_article}_{de_no}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Article {ar_ref} — MAPE {perf['MAPE']:.1f}%")

In [ ]:
# ── Cellule 7 : Prévision des 30 prochains jours ─────────────
# Pour chaque article, calculer la prévision future
# et estimer si le stock va descendre sous un seuil critique

resultats_prevision = []

for _, perf in df_metrics.iterrows():
    ar_ref = perf['AR_Ref']
    de_no  = perf['DE_No']
    
    safe_article = ar_ref.replace('/', '_').replace(' ', '_')
    model_path   = f'../models/{BASE_NAME}/{safe_article}_{de_no}.joblib'
    
    if not os.path.exists(model_path):
        continue
    
    model = joblib.load(model_path)
    
    # Stock actuel
    df_combo = df[(df['AR_Ref'] == ar_ref) & (df['DE_No'] == de_no)]
    stock_actuel = df_combo.sort_values('DateJour')['StockFinal'].iloc[-1]
    
    # Prévision future
    future   = model.make_future_dataframe(periods=HORIZON_PREVISION, freq='D')
    forecast = model.predict(future)
    prevision_future = forecast.tail(HORIZON_PREVISION)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    prevision_future['yhat']       = prevision_future['yhat'].clip(lower=0)
    prevision_future['yhat_lower'] = prevision_future['yhat_lower'].clip(lower=0)
    prevision_future['yhat_upper'] = prevision_future['yhat_upper'].clip(lower=0)
    
    # Calcul stock prévisionnel
    sorties_cumulees = prevision_future['yhat'].cumsum()
    stock_previsionnel = stock_actuel - sorties_cumulees
    
    # Jour de rupture prévu
    rupture_idx  = (stock_previsionnel <= 0).idxmax() if (stock_previsionnel <= 0).any() else None
    jours_rupture = None
    if rupture_idx is not None and stock_previsionnel[rupture_idx] <= 0:
        jours_rupture = int((prevision_future.loc[rupture_idx, 'ds'] - date_max).days)
    
    resultats_prevision.append({
        'AR_Ref':          ar_ref,
        'DE_No':           de_no,
        'stock_actuel':    round(stock_actuel, 0),
        'sorties_prev_7j': round(prevision_future['yhat'].head(7).sum(), 1),
        'sorties_prev_30j':round(prevision_future['yhat'].sum(), 1),
        'stock_prev_7j':   round(stock_actuel - prevision_future['yhat'].head(7).sum(), 1),
        'stock_prev_30j':  round(stock_actuel - prevision_future['yhat'].sum(), 1),
        'jours_rupture':   jours_rupture,
        'MAPE':            perf['MAPE']
    })

df_previsions = pd.DataFrame(resultats_prevision)

# Articles en alerte
df_alertes = df_previsions[df_previsions['jours_rupture'].notna()].sort_values('jours_rupture')

print(f"✅ Prévisions calculées pour {len(df_previsions)} articles")
print(f"\n🔴 Articles en alerte rupture ({len(df_alertes)}) :")
print(df_alertes[['AR_Ref','DE_No','stock_actuel','sorties_prev_30j','jours_rupture']].head(10).to_string(index=False))

In [ ]:
# ── Cellule 8 : Sauvegarde des résultats finaux ───────────────
df_metrics.to_csv(f'../outputs/{BASE_NAME}_metrics.csv', index=False)
df_previsions.to_csv(f'../outputs/{BASE_NAME}_previsions.csv', index=False)

print(f"✅ Métriques sauvegardées    : ../outputs/{BASE_NAME}_metrics.csv")
print(f"✅ Prévisions sauvegardées   : ../outputs/{BASE_NAME}_previsions.csv")

print(f"\n{'='*55}")
print(f"RÉCAPITULATIF FINAL — {BASE_NAME}")
print(f"{'='*55}")
print(f"Modèles entraînés  : {len(df_metrics)}")
print(f"MAPE médiane       : {df_metrics['MAPE'].median():.1f}%")
print(f"Articles alerte    : {len(df_alertes)}")
print(f"\nProchain étape : lancer FastAPI (ml/api/main.py)")
print(f"Commande       : uvicorn main:app --reload --port 8000")